In [ ]:
import requests
import pandas as pd
import os
import re
from tqdm import tqdm #barra de progreso
from dotenv import load_dotenv 

session = requests.Session()

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) #para ver toda la tabla, en nuestro caso son solo 30 se supone que no tenemos que tener problema

API_KEY = '2c6d07b262d82febbd27cf5327a36d55' 

artistas = [
    "La Fuga",
    "Héroes del Silencio",
    "Billie Eilish",
    "Love of Lesbian",
    "Estopa",
    "Mägo de Oz",
    "Mr. Kilombo",
    "Rozalén",
    "Taburete",
    "Extremoduro",
    "La Plazuela",
    "Veintiuno",
    "Ojete Calor",
    "Rata Blanca",
    "Vetusta Morla",
    "Leiva",
    "Bad Bunny",
    "Enrique Bunbury",
    "Marea",
    "Joaquín Sabina",
    "Rosalía",
    "Queen",
    "The Lumineers",
    "Foo Fighters",
    "Muse",
    "Metallica",
    "Ginebras",
    "IZAL",
    "Kaiser Chiefs",
    "Residente"
]

artistas = [a.strip() for a in artistas]

def limpiar_bio(texto):
    """Elimina etiquetas HTML como <a href=...>Read more</a>"""
    return re.sub(r'<[^>]+>', '', texto).strip()

#Funcion para llamar a la API
def obtener_info_artista(nombre):
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "artist.getinfo",
        "artist": nombre,
        "api_key": API_KEY,
        "format": "json",
        "autocorrect" : 1,  
        "lang": "es"
     }
    
    response = session.get(url, params=params, timeout=10)
    response.raise_for_status() #lanza excepción si hay error HTTP
    return response.json()

def procesar_artista(nombre):
    try:
        data = obtener_info_artista(nombre)
        artista = data["artist"]
        
        bio = limpiar_bio(artista["bio"]["content"])
        listeners = int(artista["stats"]["listeners"])
        playcount = int(artista["stats"]["playcount"])
        similares = [a["name"] for a in artista["similar"]["artist"]]
        
        return {
            "artista": nombre,
            "biografia": bio,
            "listeners": listeners,
            "playcount": playcount,
            "similares": ", ".join(similares)
        }
    
    except KeyError as e:
        print(f"[KeyError] '{nombre}': clave no encontrada {e}")  #except especifico que muestra qué clave faltó
        return None
    except session.RequestException as e:
        print(f"[RequestError] '{nombre}': {e}")
        return None

In [ ]:
resultados = []
bar = tqdm(artistas, desc="Iniciando...")
 
for artista in bar:
    bar.set_description(f"Procesando: {artista}")
    info = procesar_artista(artista)
    if info:
        resultados.append(info)
 
print(f"\n✅ {len(resultados)}/{len(artistas)} artistas obtenidos correctamente")

Procesando: Residente: 100%|██████████| 30/30 [00:05<00:00,  5.44it/s]         


✅ 30/30 artistas obtenidos correctamente


In [68]:
procesar_artista('LA fuga')

{'artista': 'LA fuga',
 'biografia': 'La Fuga son Pedro (voz y guitarra), Nando (guitarra) y Edu (batería): un grupo de Reinosa (Cantabria) con más de 500 conciertos a sus espaldas en una decena de giras, 8 discos publicados hasta la fecha, y una historia de constancia y rock and roll. Una banda cuyo esfuerzo ha situado en lo más alto del panorama de la música estatal, y cuyo comienzo se remonta a hace casi 15 años.\n\nEn el verano de 1996, más que el sofocante calor del olvidado sur de Cantabria, es la rutina como desolador panorama lo que provoca que cuatro músicos confluyan sus inquietudes en La Fuga. Rulo (voz y guitarra), Iñaky (bajo), Edu (batería) y Fito (guitarra), que ya habían buscado anteriormente una vía de escape por separado en la música, empiezan juntos a dar conciertos con versiones de grupos de rock nacionales.\n\nReinosa, conocida por su lucha obrera y sus inviernos, a cual más combativo, forjó desde el principio el carácter especial de una banda curtida con el corazó

In [62]:
#Crear DataFrame
df_lastfm = pd.DataFrame(resultados)
df_lastfm.head() #con esto veo 5

,artista,biografia,listeners,playcount,similares
0,La Fuga,"La Fuga son Pedro (voz y guitarra), Nando (gui...",82583,2800945,"Marea, Platero y tú, Rosendo, Los De Marras, F..."
1,Héroes del Silencio,Héroes es el grupo de rock español de mayor éx...,202357,6110374,"Enrique Bunbury, Caifanes, Jaguares, Fobia, Ra..."
2,Billie Eilish,Billie Eilish Pirate Baird O'Connell​ (Los Áng...,4230582,836351191,"FINNEAS, Olivia Rodrigo, Melanie Martinez, Lan..."
3,Love of Lesbian,Love of Lesbian es un grupo de pop independien...,176513,11440267,"Lori Meyers, Viva Suecia, Izal, Vetusta Morla,..."
4,Estopa,Estopa es un grupo de pop/rumba/flamenco funda...,309127,8958630,"Melendi, El Canto del Loco, Extremoduro, Perez..."


In [63]:
df_lastfm #con esto sale toda la tabla

,artista,biografia,listeners,playcount,similares
0,La Fuga,"La Fuga son Pedro (voz y guitarra), Nando (gui...",82583,2800945,"Marea, Platero y tú, Rosendo, Los De Marras, F..."
1,Héroes del Silencio,Héroes es el grupo de rock español de mayor éx...,202357,6110374,"Enrique Bunbury, Caifanes, Jaguares, Fobia, Ra..."
2,Billie Eilish,Billie Eilish Pirate Baird O'Connell​ (Los Áng...,4230582,836351191,"FINNEAS, Olivia Rodrigo, Melanie Martinez, Lan..."
3,Love of Lesbian,Love of Lesbian es un grupo de pop independien...,176513,11440267,"Lori Meyers, Viva Suecia, Izal, Vetusta Morla,..."
4,Estopa,Estopa es un grupo de pop/rumba/flamenco funda...,309127,8958630,"Melendi, El Canto del Loco, Extremoduro, Perez..."
5,Mägo de Oz,"Mägo de Oz es un grupo proveniente de Madrid, ...",298682,13967733,"Warcry, Saratoga, Saurom, Tierra Santa, Rata B..."
6,Mr. Kilombo,Mr.Kilombo es una banda mestiza nacida en Madr...,18654,219292,"Pedro Pastor, TéCanela, El Niño de La Hipoteca..."
7,Rozalén,"Desde los 7 años, María Rozalén formó parte de...",50813,808513,"Vanesa Martín, Amaral, Despistaos, Valeria Cas..."
8,Taburete,Taburete es un grupo de música madrileño forma...,38467,699817,"Maldita Nerea, Siloé, Despistaos, Antonio Oroz..."
9,Extremoduro,"En 1987 Robe, cantante y guitarra, forma Extre...",181621,12244527,"Marea, Platero y tú, Fito Y Fitipaldis, La Fug..."


In [64]:
#para verlos en orden de más popular a menos 
df_lastfm.sort_values(by="listeners", ascending=False) 

,artista,biografia,listeners,playcount,similares
21,Queen,Queen es una banda de rock clásico creada en L...,7229689,426351148,"Freddie Mercury, Roger Taylor, Brian May, Fred..."
24,Muse,Muse es una banda de rock alternativo originar...,6397894,571485208,"Royal Blood, Placebo, Nothing But Thieves, Rad..."
23,Foo Fighters,Foo Fighters es una banda de rock alternativo ...,6375398,364121042,"Pearl Jam, Taylor Hawkins & The Coattail Rider..."
25,Metallica,Metallica es una banda originaria de Estados U...,5185293,535774615,"Megadeth, Slayer, Pantera, Anthrax, Iron Maiden"
2,Billie Eilish,Billie Eilish Pirate Baird O'Connell​ (Los Áng...,4230582,836351191,"FINNEAS, Olivia Rodrigo, Melanie Martinez, Lan..."
28,Kaiser Chiefs,Kaiser Chiefs son una banda británica de Leeds...,2882998,70455540,"Kasabian, Razorlight, Hard-Fi, The Fratellis, ..."
16,Bad Bunny,"Benito Antonio Martínez Ocasio, mejor conocido...",2582126,407024693,"JHAYCO, Rauw Alejandro, Tainy, Mora, Trio Vega..."
22,The Lumineers,"Es una banda de Denver Colorado (EE.UU.) , est...",2581910,92131692,"Caamp, The Head and the Heart, Vance Joy, Mt. ..."
20,Rosalía,"Rosalía Vila Tobella, conocida mundialmente co...",1867349,203697085,"Judeline, Lorde, Robyn, Bad Gyal, NATHY PELUSO"
4,Estopa,Estopa es un grupo de pop/rumba/flamenco funda...,309127,8958630,"Melendi, El Canto del Loco, Extremoduro, Perez..."


In [65]:
# ── MEJORA: exportar a CSV para no tener que volver a llamar a la API ──
df_lastfm.to_csv("lastfm_artistas.csv", index=False, encoding="utf-8-sig")
print("✅ Datos guardados en lastfm_artistas.csv")

✅ Datos guardados en lastfm_artistas.csv


In [66]:
#merge con Deezer (cuando tengas df_deezer disponible)
df_final = df_deezer.merge(df_lastfm, on="artista", how="left")
df_final.head() 

NameError: name 'df_deezer' is not defined